In [1]:
import sys
from numbers import Real
from pathlib import Path
from html import escape

import pandas as pd
from IPython.display import HTML, display

sys.path.append("../")

from evaluation.classes import ResponseGroundednessWrapper
from evaluation.eval import groundedness, load_calibration_tests
from evaluation.reporting import save_groundedness_calibration_report


/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tests = load_calibration_tests()

In [3]:
rows = []
threshold = 0.5

for test in tests:
    score = await groundedness(
        ResponseGroundednessWrapper(
            contexts=test.contexts,
            answer=test.candidate_answer,
        )
    )
    groundedness_score = float(score) if isinstance(score, Real) else 0.0
    predicted_is_grounded = int(groundedness_score >= threshold)
    expected_is_grounded = int(test.grounded_label)

    rows.append({
        "question": test.question,
        "answer_type": test.answer_type,
        "groundedness_score": groundedness_score,
        "predicted_is_grounded": predicted_is_grounded,
        "expected_is_grounded": expected_is_grounded,
        "label_match": int(predicted_is_grounded == expected_is_grounded),
    })

report_df = pd.DataFrame(rows)
report_path = save_groundedness_calibration_report(
    report_df=report_df,
    output_path=Path("../reports/faq_groundedness_calibration_report.md"),
    threshold=threshold,
)

display(HTML(f"<p><strong>Report saved to:</strong> {escape(str(report_path))}</p>"))
report_df[[
    "question",
    "answer_type",
    "groundedness_score",
    "predicted_is_grounded",
    "expected_is_grounded",
    "label_match",
]].round({"groundedness_score": 3})

,question,answer_type,groundedness_score,predicted_is_grounded,expected_is_grounded,label_match
0,Will I get a full refund including shipping if...,adversarial,0.50,1,0,0
1,Will I get a full refund including shipping if...,bad,0.00,0,0,1
2,Will I get a full refund including shipping if...,good,1.00,1,1,1
3,Can I use multiple delivery addresses for one ...,adversarial,0.25,0,0,1
4,Can I use multiple delivery addresses for one ...,bad,0.25,0,0,1
...,...,...,...,...,...,...
145,How can I change the delivery date before my o...,bad,0.25,0,0,1
146,How can I change the delivery date before my o...,good,1.00,1,1,1
147,What should I do if my order delivery is delay...,adversarial,1.00,1,0,0
148,What should I do if my order delivery is delay...,bad,0.00,0,0,1
